# 23 — Contract Features

**Objetivo:** materializar una whitelist conservadora de atributos contractuales disponibles alrededor de firma.

No se utilizan `contracts_snapshot_state.parquet`, balances actuales, días adicionados, liquidación ni estado actual.

In [6]:
from argos.common.paths import ProjectPaths
from argos.config.gold import load_gold_spec

paths = ProjectPaths.discover()
paths.ensure_runtime_dirs()

m2_gate = paths.docs / "decisions" / "M2_GATE.md"
assert m2_gate.exists() and "M2 ✅ CLOSED" in m2_gate.read_text(encoding="utf-8"), (
    "M2 no está formalmente cerrado. No iniciar Gold."
)

GOLD_SPEC = load_gold_spec(paths.configs / "gold.toml")

{
    "gold_schema_version": GOLD_SPEC.schema_version,
    "target": GOLD_SPEC.target.name,
    "horizon_days": GOLD_SPEC.target.horizon_days,
    "event_scope": GOLD_SPEC.target.included_event_types,
    "duckdb_threads": GOLD_SPEC.runtime.duckdb_threads,
    "duckdb_memory_limit": GOLD_SPEC.runtime.duckdb_memory_limit,
    "m2_gate": str(m2_gate),
}

from argos.gold.contract_features import BUILD_IMPLEMENTATION
BUILD_IMPLEMENTATION

'm3-23-contract-features-v1'

In [7]:
from argos.gold.contract_features import build_contract_features

result = build_contract_features(
    silver_dir=paths.silver,
    gold_dir=paths.gold,
    reports_root=paths.reports,
    manifest_dir=paths.root / "artifacts" / "manifests",
    gold_spec=GOLD_SPEC,
    temp_directory=paths.root / ".cache" / "duckdb",
    overwrite=True,
)

str(result.artifact.relative_to(paths.root))

[23] materializing conservative at-signing contract feature whitelist...


'data/gold/features/contract_features.parquet'

In [8]:
result.selection

,feature,kind,available,allowed,status
0,departamento,categorical,True,True,selected
1,ciudad,categorical,True,True,selected
2,orden,categorical,True,True,selected
3,sector,categorical,True,True,selected
4,rama,categorical,True,True,selected
5,entidad_centralizada,categorical,True,True,selected
6,codigo_de_categoria_principal,categorical,True,True,selected
7,tipo_de_contrato,categorical,True,True,selected
8,modalidad_de_contratacion,categorical,True,True,selected
9,justificacion_modalidad_de_contratacion,categorical,True,True,selected


In [9]:
result.validation

,check_id,category,blocking,passed,observed,requirement
0,contract_feature_population_preserved,grain,True,True,"{""distinct_contracts"": 5640278, ""expected"": 56...",one contract-feature row per contract with val...
1,contract_feature_t0_complete,temporal,True,True,"{""null_t0"": 0}",contract feature rows always carry their predi...
2,contract_forbidden_columns_absent,leakage,True,True,"{""forbidden_present"": []}",M1/M2 mutable snapshot fields are absent from ...
3,contract_feature_whitelist_sufficient,features,True,True,"{""derived_calendar"": 4, ""selected_configured"":...",at least 5 conservative features are materialized


In [10]:
blocking = result.validation.loc[result.validation["blocking"]]
print("23 ✅ PASS" if result.ready_for_next_notebook else "23 ❌ FAIL")
print(f"Blocking checks: {int(blocking['passed'].sum())}/{len(blocking)}")
print("Manifest:", result.manifest_path)
assert result.ready_for_next_notebook, "Contract features no superan sus checks bloqueantes."

23 ✅ PASS
Blocking checks: 4/4
Manifest: /Users/macm1pro/Desktop/projects/argos/argos-code/artifacts/manifests/gold_contract_features.json
